# Beginner Tutorial: Otsu Thresholding + Classical Image Segmentation

This notebook combines ideas from `01_otsu_thresholding.ipynb` and `Image_Segmentation.ipynb` into one guided workflow.

## Audience
This tutorial assumes **no prior AI/ML background**. We explain each idea before coding it.

## What you will learn
1. What threshold-based segmentation means.
2. How Otsu, Niblack, and Sauvola thresholding differ.
3. How to process multiple XCT slices as a stack.
4. How to save segmented (binary) outputs.
5. (Optional) How HMRF can improve segmentation by using spatial context.

## Step 0 - Imports and setup
We import the tools used throughout the notebook.

If `pore2chip` is not installed, the notebook still works with scikit-image methods.

In [ ]:
# ----------------------------
# Basic imports
# ----------------------------
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from PIL import Image
from skimage.filters import threshold_otsu, threshold_niblack, threshold_sauvola

# Optional package from 01_otsu_thresholding.ipynb
try:
    from pore2chip import filter_im
    HAS_PORE2CHIP = True
except Exception:
    HAS_PORE2CHIP = False

# Local HMRF helper module from this folder
import HMRF_EM as hmrf

print("pore2chip available:", HAS_PORE2CHIP)

## Step 1 - Helper functions
Before we start processing images, we define tiny reusable helper functions.

This follows the same teaching style as `randomforest-imgseg.ipynb`: short, modular code blocks.

In [ ]:
# ----------------------------
# Helper functions
# ----------------------------
def to_grayscale(arr):
    """Convert image arrays to 2D grayscale if needed."""
    if arr.ndim == 3:
        return np.mean(arr[..., :3], axis=2)
    return arr

def load_grayscale_slice(path):
    """Read one slice and return float grayscale array."""
    return to_grayscale(mpimg.imread(path)).astype(np.float32)

## Step 2 - Configuration
This step entails downloading the toy data and configuring directories

In [ ]:
!git clone https://github.com/aramyxt/MONet_Pore2Chip_data.git

In [ ]:
# ----------------------------
# Configuration
# ----------------------------
# Main image folder for XCT slices.
IMAGE_DIR = "./MONet_Pore2Chip_data/data/bean_bucket_100/"

# Number of slices to process in the stack example.
MAX_SLICES_TO_LOAD = 10

# File extensions accepted by the loader.
VALID_EXTS = (".tif", ".tiff", ".png")

# Local thresholding parameters (from Image_Segmentation.ipynb-style methods).
NIBLACK_WINDOW = 101
NIBLACK_K = 0.2
SAUVOLA_WINDOW = 101

## Step 3 - Load one XCT slice and inspect intensity
Before segmentation, we inspect one grayscale slice and its histogram.

Why this matters: thresholding splits pixels by intensity, so the histogram helps us predict whether a clean split is likely.

In [ ]:
# ----------------------------
# Load one sample image
# ----------------------------
# Discover input images using configured folder and valid extensions.
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(VALID_EXTS)])

assert len(image_files) > 0, "No images found in IMAGE_DIR"

sample_file = image_files[0]
sample_path = os.path.join(IMAGE_DIR, sample_file)
sample_img = load_grayscale_slice(sample_path)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(sample_img, cmap="gray")
ax[0].set_title(f"Sample slice: {sample_file}")
ax[0].axis("off")

ax[1].hist(sample_img.ravel(), bins=80)
ax[1].set_title("Intensity histogram")
ax[1].set_xlabel("Pixel intensity")
ax[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Step 4 - Compare thresholding methods (Otsu, Niblack, Sauvola)
This section follows the thresholding comparison from `Image_Segmentation.ipynb`.

- **Otsu**: one global threshold for the whole image
- **Niblack**: local threshold based on local mean and variance
- **Sauvola**: local threshold, often more robust to uneven contrast

In [ ]:
# ----------------------------
# Threshold comparison on one slice
# ----------------------------
thresh_otsu = threshold_otsu(sample_img)
thresh_niblack = threshold_niblack(sample_img, window_size=NIBLACK_WINDOW, k=NIBLACK_K)
thresh_sauvola = threshold_sauvola(sample_img, window_size=SAUVOLA_WINDOW)

binary_otsu = sample_img > thresh_otsu
binary_niblack = sample_img > thresh_niblack
binary_sauvola = sample_img > thresh_sauvola

fig, ax = plt.subplots(2, 2, figsize=(11, 11))
ax[0, 0].imshow(sample_img, cmap="gray")
ax[0, 0].set_title("Original image")
ax[0, 0].axis("off")

ax[0, 1].imshow(binary_otsu, cmap="gray")
ax[0, 1].set_title(f"Otsu (global), t={thresh_otsu:.3f}")
ax[0, 1].axis("off")

ax[1, 0].imshow(binary_niblack, cmap="gray")
ax[1, 0].set_title("Niblack (local)")
ax[1, 0].axis("off")

ax[1, 1].imshow(binary_sauvola, cmap="gray")
ax[1, 1].set_title("Sauvola (local)")
ax[1, 1].axis("off")

plt.tight_layout()
plt.show()

## Step 5 - Otsu segmentation across multiple slices (stack processing)
Now we move from one image to a stack of XCT slices, inspired by `01_otsu_thresholding.ipynb`.

Key idea: each slice is segmented independently, then stored in a 3D array.

In [ ]:
# ----------------------------
# Load a stack of grayscale slices
# ----------------------------
NUM_SLICES = min(MAX_SLICES_TO_LOAD, len(image_files))

stack_list = []
for fname in image_files[:NUM_SLICES]:
    arr = load_grayscale_slice(os.path.join(IMAGE_DIR, fname))
    stack_list.append(arr)

img_stack = np.stack(stack_list, axis=0)
print("Loaded stack shape (slices, H, W):", img_stack.shape)

plt.figure(figsize=(4, 4))
plt.imshow(img_stack[0], cmap="gray")
plt.title("First slice in stack")
plt.axis("off")
plt.show()

In [ ]:
# ----------------------------
# Apply Otsu per slice
# ----------------------------
def otsu_stack_threshold(stack):
    """Apply Otsu threshold independently to each 2D slice."""
    binary_stack = np.zeros_like(stack, dtype=np.uint8)
    thresholds = []

    for i in range(stack.shape[0]):
        t = threshold_otsu(stack[i])
        thresholds.append(float(t))
        binary_stack[i] = (stack[i] > t).astype(np.uint8)

    return binary_stack, thresholds

binary_stack_otsu, otsu_thresholds = otsu_stack_threshold(img_stack)
print("First 5 Otsu thresholds:", otsu_thresholds[:5])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(img_stack[0], cmap="gray")
ax[0].set_title("Original slice")
ax[0].axis("off")

ax[1].imshow(binary_stack_otsu[0], cmap="gray")
ax[1].set_title("Otsu binary slice")
ax[1].axis("off")

plt.tight_layout()
plt.show()

## Step 6 - Save thresholded outputs
This reproduces the save-to-disk behavior from `01_otsu_thresholding.ipynb`.

Each segmented slice is saved as a PNG image with values 0 (background) and 255 (foreground).

In [ ]:
# ----------------------------
# Save Otsu binary slices
# ----------------------------
OUT_DIR_OTSU = "./filtered_images_otsu_combined"
os.makedirs(OUT_DIR_OTSU, exist_ok=True)

for i in range(binary_stack_otsu.shape[0]):
    out_path = os.path.join(OUT_DIR_OTSU, f"slice_{i:03d}.png")
    Image.fromarray((binary_stack_otsu[i] * 255).astype(np.uint8)).save(out_path)

print("Saved Otsu masks to:", OUT_DIR_OTSU)

## Step 7 (Optional) - Use Pore2Chip shortcuts
This section mirrors both methods from `01_otsu_thresholding.ipynb`:
1. Filter a stack already loaded in memory.
2. Read and filter directly from an image folder.

If `pore2chip` is missing, this cell safely skips.

In [ ]:
# ----------------------------
# Optional Pore2Chip workflow
# ----------------------------
if HAS_PORE2CHIP:
    # Method 1: filter from stack already in memory
    # Invert=True matches the original notebook behavior.
    p2c_filtered_from_memory = filter_im.filter_list(img_stack.astype(np.uint8), invert=True)

    # Method 2: read and filter directly from folder
    p2c_filtered_from_path = filter_im.read_and_filter_list(IMAGE_DIR, crop_depth=NUM_SLICES, invert=True)

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(p2c_filtered_from_memory[0], cmap="gray")
    ax[0].set_title("Pore2Chip: filter_list")
    ax[0].axis("off")

    ax[1].imshow(p2c_filtered_from_path[0], cmap="gray")
    ax[1].set_title("Pore2Chip: read_and_filter_list")
    ax[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Skipping Pore2Chip section because pore2chip is not installed.")

## Step 8 (Concept) - U-Net architecture image and how it works
U-Net is a deep learning segmentation model. Even though this notebook focuses on thresholding, this diagram helps learners connect to modern AI methods.

How to read the architecture:
- **Encoder (left)** downsamples and learns increasingly abstract features.
- **Bottleneck (middle)** stores compact high-level context.
- **Decoder (right)** upsamples to pixel-level predictions.
- **Skip connections** copy fine details from encoder to decoder, improving mask boundaries.

In [ ]:
# ----------------------------
# U-Net concept diagram
# ----------------------------
# This draws a simple architecture image directly in the notebook so learners
## can visually map encoder -> bottleneck -> decoder with skip connections.
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis("off")

def block(x, y, w, h, text, color):
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.15", facecolor=color, edgecolor="black")
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=9)

# Encoder blocks (downsampling path).
block(0.6, 3.8, 1.6, 1.0, "Enc 1\n(High res)", "#d9edf7")
block(2.8, 3.1, 1.6, 1.0, "Enc 2", "#d9edf7")
block(5.0, 2.4, 1.6, 1.0, "Bottleneck", "#fdebd0")

# Decoder blocks (upsampling path).
block(7.2, 3.1, 1.6, 1.0, "Dec 2", "#d5f5e3")
block(9.4, 3.8, 1.6, 1.0, "Dec 1\n(Output mask)", "#d5f5e3")

# Main forward arrows.
for x1, y1, x2, y2 in [(2.2, 4.3, 2.8, 3.6), (4.4, 3.6, 5.0, 2.9), (6.6, 2.9, 7.2, 3.6), (8.8, 3.6, 9.4, 4.3)]:
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="->", mutation_scale=12, linewidth=1.5))

# Skip connections.
ax.add_patch(FancyArrowPatch((2.2, 4.6), (9.4, 4.6), arrowstyle="-|>", mutation_scale=12, linewidth=1.4, linestyle="--"))
ax.add_patch(FancyArrowPatch((4.4, 3.9), (7.2, 3.9), arrowstyle="-|>", mutation_scale=12, linewidth=1.4, linestyle="--"))

ax.text(6.0, 5.4, "U-Net: Encoder + Decoder + Skip Connections", ha="center", fontsize=12, weight="bold")
ax.text(6.0, 0.8, "Dashed arrows are skip connections that preserve fine spatial details.", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## Step 9 (Optional Advanced) - HMRF segmentation on one slice
HMRF (Hidden Markov Random Field) is more advanced than plain thresholding.

Beginner intuition:
- Otsu uses only pixel intensity values.
- HMRF uses intensity **and** nearby-pixel agreement (spatial context).

This often reduces noisy isolated pixels.

In [ ]:
# ----------------------------
# HMRF example on one slice
# ----------------------------
# Compatibility patch for NumPy versions where np.bool is removed.
if not hasattr(np, "bool"):
    np.bool = bool  # type: ignore[attr-defined]

slice_2d = img_stack[0]
data = slice_2d.reshape(-1, 1)
num_samples = data.shape[0]
num_clusters = 2
num_dims = 1
img_size_for_hmrf = (1, slice_2d.shape[0], slice_2d.shape[1])

hmrf_label, hmrf_energy, hmrf_params = hmrf.EM_calc(
    num_dims=num_dims,
    num_samples=num_samples,
    num_clusters=num_clusters,
    data=data,
    img_size=img_size_for_hmrf,
    void_weight=1.0,
    beta=1.0,
    verbose=False,
)

hmrf_mask = hmrf_label.reshape(slice_2d.shape).astype(np.uint8)
otsu_mask_single = (slice_2d > threshold_otsu(slice_2d)).astype(np.uint8)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(slice_2d, cmap="gray")
ax[0].set_title("Original")
ax[0].axis("off")

ax[1].imshow(otsu_mask_single, cmap="gray")
ax[1].set_title("Otsu result")
ax[1].axis("off")

ax[2].imshow(hmrf_mask, cmap="gray")
ax[2].set_title("HMRF result")
ax[2].axis("off")

plt.tight_layout()
plt.show()

## Wrap-up
You now have one notebook that combines:
- Basic Otsu thresholding workflow
- Local thresholding comparison (Niblack, Sauvola)
- Multi-slice stack processing and export
- Optional Pore2Chip shortcuts
- U-Net architecture concept and visual diagram
- Optional advanced HMRF segmentation

### Suggested beginner exercises
1. Change Niblack/Sauvola window size and compare results.
2. Increase `MAX_SLICES_TO_LOAD` and check runtime vs quality.
3. Try different `beta` values in HMRF and observe smoothness changes.